# Notebooks, a Python recap and NumPy

You know the basics of Python from the first semester. This block makes sure everyone works in the same environment, recaps the Python you need in a few minutes, and then introduces **NumPy**, the package for computing with whole arrays of numbers at once. Almost every data package of the course (pandas, scikit-learn, PyTorch) stores its numbers in NumPy arrays or in arrays that behave the same way, so the ideas of this block come back in every session.

The examples use the Berlin Airbnb listings and their booking calendar (Inside Airbnb, snapshot of 26 June 2026). pandas, the topic of the next block, is used here only to read the files.

```mermaid
flowchart LR
  E["Environment<br/>uv, VS Code, .venv"] --> N["Notebook<br/>cells and kernel"]
  N --> R["Python recap<br/>types, collections,<br/>loops, functions"]
  R --> A["NumPy arrays<br/>dtype, shape,<br/>indexing, masks"]
  A --> V["Vectorised operations<br/>broadcasting,<br/>aggregation by axis"]
  V --> S["Random numbers<br/>and simulation"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## The working environment: uv, VS Code and the course environment

A data analysis needs a Python interpreter and the **packages** it imports. Different projects may need different versions of the same package, so each project gets a **virtual environment**: a folder with one interpreter and one set of packages.

| Tool | What it does in the course |
|---|---|
| **uv** | Installs Python and the packages. `uv sync` creates the course environment in `.venv`; `uv run <command>` runs a command inside it. |
| **VS Code** | The editor. It opens notebooks (`.ipynb`) and Python files (`.py`) and runs them with the interpreter of `.venv`. |
| **Git** | Downloads the course material and later keeps the history of your team's code (Session 2). |

`pyproject.toml` lists the packages the course needs; `uv.lock` records the exact version of each one. Because everyone installs from the same lock file, a notebook that runs on the lecturer's laptop also runs on yours. The course has **one environment** for all sessions: the kernel picker at the top right of a notebook must show `.venv`.

In [2]:
import sys

import numpy as np
import pandas as pd

print(sys.version.split()[0])                 # the Python version, 3.14.x in the course
print(Path(sys.executable).parents[1].name)   # the environment folder: .venv
print("NumPy", np.__version__, "| pandas", pd.__version__)

3.14.7
.venv
NumPy 2.5.4 | pandas 3.0.6


If the second line does not print `.venv`, the notebook runs with another Python and packages may be missing: choose `.venv` in the kernel picker. `import numpy as np` and `import pandas as pd` are the conventional short names, used in every tutorial and in the documentation.

> **In practice.** "It works on my machine" is one of the most common problems in data teams. A lock file makes the environment reproducible; Session 2 uses the same `uv.lock` to run a team's tests on a fresh machine on every change.

## Jupyter notebooks: cells, kernel, Markdown, order of execution

A notebook is a list of **cells**. A Markdown cell holds text; a code cell holds Python, and running it (Shift+Enter) sends the code to the **kernel**, a running Python process that keeps every name in memory. The value of the last expression of a cell is displayed; the number to the left, such as `[3]`, counts the order in which cells were run. The [Think Python introduction](02-jupyter-introduction.ipynb) shows the buttons and modes step by step.

**Order of execution.** The kernel runs cells in the order *you* run them, not the order on the page. If you run a cell further down that changes `price`, then rerun a cell above, the result no longer matches what the page shows. A notebook that only works because of such a hidden history is not reproducible.

> **Rule.** Before you share a notebook or trust its result, choose *Restart* and then *Run All*.

Useful keys after Esc (command mode): **A** or **B** inserts a cell above or below, **M** turns it into Markdown, **Y** into code, **D D** deletes it. Markdown itself is short: `## Heading`, `**bold**`, `*italic*`, `- item`, `` `code` `` and `[pandas](https://pandas.pydata.org)` for a link.

## A short recap of Python: types, collections, conditions, loops and functions

This is a reminder, not a course. If a line below is unfamiliar, work through the self-study notebooks linked at the end of this section.

| Idea | Remember |
|---|---|
| `int`, `float`, `str`, `bool`, `None` | Every value has a type. A missing price is `None`, not `0`; a postal code such as `"01067"` stays text. `0.1 + 0.2` is not exactly `0.3`. |
| `list`, `tuple`, `dict`, `set` | Ordered and changeable; ordered and fixed; records with named fields; unique values. A list of dictionaries is a first table. |
| `=` | Binds a name to a value once; it is not a formula. Two names can refer to the **same** list. |
| `if` / `elif` / `else` | Only the first true branch runs, so the order of the conditions matters. |
| `for`, `while`, comprehensions | `for` with an accumulator processes each item; `[f(x) for x in items if ...]` builds a list in one line. |
| `def` ... `return` | A function gives a rule a name; `return` hands back a value, `print` only shows it. |

The cell below uses all of them on three invented listings:

In [3]:
records = [
    {"district": "Pankow", "nightly_price": 75, "available": True},
    {"district": "Mitte", "nightly_price": 90, "available": False},
    {"district": "Neukölln", "nightly_price": None, "available": True},   # price not recorded
]


def stay_cost(nightly_price: float, nights: int, cleaning_fee: float = 25) -> float:
    """Return the total cost of a stay; the cleaning fee is paid once."""
    return nightly_price * nights + cleaning_fee


affordable = []                                   # accumulator
for r in records:
    if r["nightly_price"] is None or not r["available"]:
        continue                                  # skip missing prices and booked flats
    if stay_cost(r["nightly_price"], nights=3) <= 300:
        affordable.append(r["district"])

prices = [r["nightly_price"] for r in records if r["nightly_price"] is not None]
print(affordable, prices, stay_cost(75, 3))

['Pankow'] [75, 90] 250


The loop works, but it handles one value at a time in Python. With 6,701 prices instead of three, and with many operations in a row, that becomes slow and long. The next cell compares a Python list comprehension with the same computation on a NumPy array (`%timeit` runs a line many times and reports the average; the times depend on the machine).

In [4]:
listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
short = listings["price"].notna() & (listings["minimum_nights"] < 28)   # short stays with a price
price_list = listings.loc[short, "price"].tolist()                       # a Python list of 6,701 floats
price_array = np.array(price_list)                                       # the same values as a NumPy array

%timeit [p * 3 + 25 for p in price_list]
%timeit price_array * 3 + 25

204 μs ± 12.1 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


3.49 μs ± 61.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


Both lines compute the cost of a three-night stay for every listing. The array version has no loop in Python: NumPy runs the loop in compiled code, and on most laptops it is more than ten times faster. It is also shorter and reads like the formula. The rest of this block is about working this way.

> 📖 **Read more:** the Python self-study notebooks from *A Whirlwind Tour of Python* (CC0): [variables and references](extras/01-python-variables.ipynb), [scalar types](extras/02-python-scalar-types.ipynb), [lists, tuples, dictionaries and sets](extras/03-python-data-structures.ipynb), [control flow](extras/04-python-control-flow.ipynb), [functions](extras/05-python-functions.ipynb), [errors and exceptions](extras/06-python-errors-and-exceptions.ipynb) and [modules and packages](extras/07-python-modules-and-packages.ipynb). Work through them if the recap went too fast.

## NumPy arrays: creation, `dtype` and `shape`

A NumPy **array** (`ndarray`) is a grid of values that all have the **same type**, its `dtype`. Because every element has the same size in memory, NumPy can process the whole grid in compiled code. An array has a **shape**: a tuple with the length along each **axis**. A one-dimensional array of 5 prices has shape `(5,)`; a table of 3 listings by 4 nights has shape `(3, 4)`, three rows along axis 0 and four columns along axis 1.

| Create | Example | Result |
|---|---|---|
| from a list | `np.array([75, 90, 110])` | `[75 90 110]`, dtype `int64` |
| a range | `np.arange(1, 8)` | the nights 1 to 7 |
| evenly spaced | `np.linspace(0, 1, 5)` | `[0. 0.25 0.5 0.75 1.]` |
| filled | `np.zeros((3, 4))`, `np.ones(5)`, `np.full(7, 25.0)` | arrays of 0, 1 or 25 |
| reshape | `np.arange(12).reshape(3, 4)` | the same 12 values as 3 rows of 4 |

In [5]:
prices = np.array([75, 90, 110, 45, 160])
print(prices, prices.dtype, prices.shape, prices.ndim, prices.size)

with_decimals = np.array([75, 90.5, 110])          # one float makes every element a float
print(with_decimals, with_decimals.dtype)

nights = np.arange(1, 8)
print(nights)

table = np.arange(12).reshape(3, 4)                # 3 rows (axis 0), 4 columns (axis 1)
print(table)
print(table.shape)

[ 75  90 110  45 160] int64 (5,) 1 5
[ 75.   90.5 110. ] float64
[1 2 3 4 5 6 7]
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(3, 4)


A column of a pandas table is a NumPy array (or behaves like one): `.to_numpy()` returns it. The price column of the listings is `float64`, because NumPy has no integer value for "missing": a missing price is stored as the special float **`NaN`** ("not a number").

In [6]:
price = listings["price"].to_numpy()
print(price.dtype, price.shape)
print(price[:5])
print(np.isnan(price).sum(), "missing prices")

float64 (12776,)
[160.71 193.33    nan  10.88 372.67]
4335 missing prices


> **In practice.** Look at `dtype` and `shape` first whenever a result surprises you. A price column with dtype `object` or `str` holds text (for example `"$75.00"`) and cannot be used in arithmetic; a shape of `(12776, 1)` where you expected `(12776,)` breaks broadcasting later.

> 📖 **Read more:** [extras/08-numpy-array-basics.ipynb](extras/08-numpy-array-basics.ipynb) (Python Data Science Handbook): attributes, indexing, slicing, reshaping, joining and splitting arrays, with many small examples.

## Indexing, slicing and Boolean masks

Indexing works as for lists, starting at 0, and negative positions count from the end. A two-dimensional array takes one index per axis: `table[row, column]`. A **slice** `start:stop` stops before `stop`, and `:` alone means "everything along this axis".

In [7]:
print(prices[0], prices[-1], prices[1:3])
print(table[1, 2])          # row 1, column 2
print(table[0])             # the first row
print(table[:, 0])          # the first column
print(table[:2, 1:3])       # rows 0-1, columns 1-2

75 160 [ 90 110]
6
[0 1 2 3]
[0 4 8]
[[1 2]
 [5 6]]


**A slice is a view, not a copy.** Slicing a list creates a new list; slicing an array returns a *view* on the same memory. Changing the view changes the original array. Call `.copy()` when you need an independent array.

In [8]:
first_two = prices[:2]
first_two[0] = 999
print(prices)              # the original changed, too
prices[0] = 75             # restore it

[999  90 110  45 160]


**Boolean masks.** A comparison on an array gives an array of `True` and `False` of the same shape, a **mask**. Indexing with a mask keeps the elements where it is `True`. Masks combine with `&` (and), `|` (or) and `~` (not); each comparison needs parentheses, because `&` binds more strongly than `<`. Python's `and` and `or` work on single values and raise an error on arrays.

`True` counts as 1 and `False` as 0, so `mask.sum()` counts the matches and `mask.mean()` gives their share.

In [9]:
minimum_nights = listings["minimum_nights"].to_numpy()

has_price = ~np.isnan(price)
short_stay = has_price & (minimum_nights < 28)
print(has_price.sum(), short_stay.sum(), short_stay.mean().round(3))

short_price = price[short_stay]
print(short_price.shape, np.median(short_price), short_price.mean().round(2))
print((short_price > 500).sum(), "short-stay listings ask more than €500 a night")

8441 6701 0.524
(6701,) 157.0 193.69
180 short-stay listings ask more than €500 a night


6,701 listings are short stays with a price, 52 % of all listings. Their median price is €157 a night, the mean about €194: a few very expensive listings pull the mean up.

> **In practice.** A mask is how you define a population. Say which rows you counted ("short stays with a price") whenever you report a number; the next block uses the same masks on pandas tables.

## Vectorised operations and broadcasting

Arithmetic and comparisons on arrays work **element by element**: `short_price * 3 + 25` multiplies every price and adds the fee to every result. NumPy's **universal functions** (`np.sqrt`, `np.log`, `np.round`, `np.maximum`, ...) do the same for mathematical functions. Code without explicit Python loops is called **vectorised**.

**Missing values propagate.** Any arithmetic with `NaN` gives `NaN`, and so does `np.median` of an array that contains one. Remove the missing values with a mask, or use the `nan`-functions (`np.nanmedian`, `np.nanmean`, `np.nansum`), which skip them.

In [10]:
three_nights = short_price * 3 + 25
print(three_nights[:3])
print(np.log10(short_price[:3]).round(2))       # a universal function

print(np.median(price), np.nanmedian(price))    # NaN propagates; the nan-version skips it

[ 507.13  604.99 1143.01]
[2.21 2.29 2.57]
nan 130.67


**Broadcasting** extends element-wise arithmetic to arrays of different shapes. NumPy compares the shapes from the right; two lengths fit if they are equal or one of them is 1, and an axis of length 1 is stretched to match the other. A column of 3 prices with shape `(3, 1)` times a row of 7 nights with shape `(7,)` gives a full `(3, 7)` table, without writing a loop and without copying the data.

```mermaid
flowchart LR
  P["prices[:, np.newaxis]<br/>shape (3, 1)"] --> T["P * nights + fee<br/>shape (3, 7)"]
  N["nights<br/>shape (7,)"] --> T
  F["fees[:, np.newaxis]<br/>shape (3, 1)"] --> T
```

`np.newaxis` (or `.reshape(-1, 1)`) turns a one-dimensional array into a column.

In [11]:
nightly = np.array([75.0, 110.0, 160.0])       # three listings
fees = np.array([25.0, 40.0, 60.0])            # their cleaning fees
nights = np.arange(1, 8)                       # stays of 1 to 7 nights

cost = nightly[:, np.newaxis] * nights + fees[:, np.newaxis]
print(cost.shape)
print(cost)
print(cost <= 500)                             # which stays fit a budget of €500?

(3, 7)
[[ 100.  175.  250.  325.  400.  475.  550.]
 [ 150.  260.  370.  480.  590.  700.  810.]
 [ 220.  380.  540.  700.  860. 1020. 1180.]]
[[ True  True  True  True  True  True False]
 [ True  True  True  True False False False]
 [ True  True False False False False False]]


Row 0 is the €75 flat: €100 for one night, €550 for seven. If the shapes do not fit, NumPy stops with an error such as `operands could not be broadcast together with shapes (3,) (7,)`: here `nightly * nights` without `np.newaxis` fails, because 3 and 7 are neither equal nor 1.

> **In practice.** Broadcasting is how a model subtracts the mean of every column from a whole data table at once (`X - X.mean(axis=0)`), which Session 6 uses for scaling. When a broadcast result has an unexpected shape, print the shapes of both operands.

> 📖 **Read more:** [extras/09-numpy-ufuncs.ipynb](extras/09-numpy-ufuncs.ipynb) (Python Data Science Handbook): the universal functions, from arithmetic to exponents, logarithms and special functions.
>
> 📖 **Read more:** [extras/10-numpy-broadcasting.ipynb](extras/10-numpy-broadcasting.ipynb) (Python Data Science Handbook): the three broadcasting rules with diagrams and worked cases, including centring an array.

## Aggregation along an axis

Aggregations reduce many values to one: `sum`, `mean`, `median`, `min`, `max`, `std`, and `argmin`/`argmax` for the *position* of the smallest or largest value. On a two-dimensional array the `axis` argument says which axis disappears.

```mermaid
flowchart LR
  A["array, shape (listings, days)"] -->|"sum(axis=1)<br/>collapse the days"| B["one value per listing<br/>shape (listings,)"]
  A -->|"mean(axis=0)<br/>collapse the listings"| C["one value per day<br/>shape (days,)"]
  A -->|"sum()<br/>no axis"| D["one number"]
```

The booking calendar of the case study has one row per listing and day: whether the night is still available, for the 365 days that start on the day the listing was scraped. A night that is not available is booked or blocked by the host; the calendar does not say which. Sorting the rows by listing and date and reshaping the column gives a two-dimensional Boolean array: one row per listing, one column per day ahead.

In [12]:
calendar = pd.read_parquet("case-study/data/airbnb/calendar.parquet")
calendar = calendar.sort_values(["listing_id", "date"])
n_listings = calendar["listing_id"].nunique()

available = calendar["available"].to_numpy().reshape(n_listings, 365)
print(available.shape, available.dtype)
print(available[:3, :10].astype(int))          # three listings, first ten days

(12855, 365) bool
[[0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 1 1 0]
 [0 0 0 1 1 1 1 1 1 1]]


The `reshape` is only correct because every listing has exactly 365 rows, which the case-study script guarantees; on other data check this first (`calendar.groupby("listing_id").size()`, next block). The calendar has 12,855 listings, a few more than the 12,776 in the listings table.

In [13]:
nights_open = available.sum(axis=1)            # per listing: available nights in the next year
print(nights_open.shape, np.median(nights_open), nights_open.mean().round(1))
print((nights_open == 0).mean().round(3), "of the listings have no available night at all")

share_open = available.mean(axis=0)            # per day ahead: share of listings still available
print(share_open.shape)
print(share_open[[0, 7, 30, 90, 180]].round(3))
print("most open:", share_open.argmax(), "days ahead, share", share_open.max().round(3))

(12855,) 137.0 152.9
0.271 of the listings have no available night at all
(365,)
[0.083 0.254 0.372 0.227 0.441]
most open: 172 days ahead, share 0.498


The median listing has 137 available nights in the coming year, and 27 % have none at all: they are fully booked or, much more likely, blocked by hosts who do not rent them out at the moment. On the day of the scrape only 8 % of the listings are still available, a week ahead 25 % and a month ahead 37 %: nearby nights are already booked. Further ahead the share stays between about 35 % and 50 %, with a few short dips (23 % about 90 days ahead) and a maximum of 50 % about 172 days ahead. One line per question, and no loop.

> **In practice.** Before you aggregate, say which axis you collapse and what one value of the result means ("one value per listing", "one value per day"). Most axis mistakes produce an array of the wrong length, which `shape` reveals immediately.

## Random numbers and simulation

Simulation answers questions that are hard to compute by hand: roll the dice, or draw the sample, many times and count. NumPy's random numbers come from a **generator** created with `np.random.default_rng(seed)`. The **seed** fixes the sequence, so the notebook gives the same result on every run and on every laptop; without a seed, every run differs.

| Method | Draws |
|---|---|
| `rng.integers(1, 7, size=10)` | 10 whole numbers from 1 to 6 (the upper bound is excluded): ten rolls of a die |
| `rng.random(5)` | 5 floats between 0 and 1 |
| `rng.normal(loc, scale, size)` | values from a normal distribution |
| `rng.choice(values, size, replace=False)` | a random sample from an array |

In [14]:
rng = np.random.default_rng(42)
print(rng.integers(1, 7, size=10))

# The share of sixes approaches 1/6 = 0.167 as the number of rolls grows.
for n in [10, 100, 10_000, 1_000_000]:
    rolls = rng.integers(1, 7, size=n)
    print(f"{n:>9,} rolls: share of sixes {(rolls == 6).mean():.3f}")

[1 5 4 3 3 6 1 5 2 1]
       10 rolls: share of sixes 0.200
      100 rolls: share of sixes 0.110
   10,000 rolls: share of sixes 0.163


1,000,000 rolls: share of sixes 0.166


Small samples scatter widely around 1/6; a million rolls are close to it (the *law of large numbers*). The same idea tells us how much a statistic computed from a sample can vary. Suppose we could only check 50 random short-stay listings: how far would their median price be from the median of all 6,701?

In [15]:
medians = np.array([np.median(rng.choice(short_price, size=50, replace=False)) for _ in range(1_000)])
print(f"median of all short stays: {np.median(short_price):.0f}")
print(f"medians of 1,000 samples of 50: from {medians.min():.0f} to {medians.max():.0f}")
print(f"middle 90 %: {np.percentile(medians, 5):.0f} to {np.percentile(medians, 95):.0f}")

median of all short stays: 157
medians of 1,000 samples of 50: from 118 to 222
middle 90 %: 134 to 185


One sample in ten gives a median below €134 or above €185, more than €20 away from the €157 of all short stays. Session 5 turns this simulation into the bootstrap and confidence intervals.

> **In practice.** Always pass a seed to the generator in analysis code and in the final project, and say how many repetitions a simulation used. Do not use the older `np.random.seed` and `np.random.rand` functions in new code; the NumPy documentation recommends the generator.

## Check your understanding

1. You run a cell that sets `price = 95`, then a cell above it that computes `price * 3`. What does the page show, and how do you protect yourself from such surprises?
2. What are the `dtype` and the `shape` of `np.array([[1, 2, 3], [4, 5, 6.0]])`?
3. `a = np.arange(5); b = a[1:3]; b[0] = 99`. What is `a` now, and how would you avoid this?
4. An array `X` has shape `(1000, 8)`. What are the shapes of `X.mean(axis=0)`, `X.mean(axis=1)` and `X - X.mean(axis=0)`?
5. Why does `np.median(price)` return `nan` for the listings, and what are two ways to get the median of the prices that are present?

## Further reading

- NumPy developers (2026). *NumPy: the absolute basics for beginners*. https://numpy.org/doc/stable/user/absolute_beginners.html
- NumPy developers (2026). *Broadcasting* and *Random sampling* (the `Generator` interface). https://numpy.org/doc/stable/user/basics.broadcasting.html, https://numpy.org/doc/stable/reference/random/index.html
- McKinney, W. (2022). *Python for Data Analysis*, 3rd edition, chapter 4 (NumPy basics: arrays and vectorised computation). Open access. https://wesmckinney.com/book/
- VanderPlas, J. (2016). *Python Data Science Handbook*, chapter 2 (introduction to NumPy). O'Reilly. https://jakevdp.github.io/PythonDataScienceHandbook/ (the source of extras 08–10).
- Python Software Foundation (2026). *The Python Tutorial*, sections 3–5. https://docs.python.org/3/tutorial/
- Project Jupyter (2026). *JupyterLab documentation: notebooks*. https://jupyterlab.readthedocs.io/en/stable/user/notebook.html
- Astral (2026). *uv documentation: working on projects*. https://docs.astral.sh/uv/guides/projects/